In [29]:
# [CELL 1] IMPORT REQUIRED LIBRARIES

import csv
import json
import shutil
from pathlib import Path
from collections import Counter
import re


In [30]:
# [CELL 2] STUDENT-SPECIFIC CONFIGURATION

LAST_NAME = "Correa"
STUDENT_ID = "TUPM-26-1106"

SEED_DIGIT = 6
ID_SUM = 16

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Records to process:", RECORDS_TO_PROCESS)
print("Target status class:", f"{TARGET_STATUS_CLASS}xx")

Student: Correa
Student ID: TUPM-26-1106
Records to process: 1016
Target status class: 4xx


In [31]:
# [CELL 3] CREATE DIRECTORIES AND DEFINE FILE PATHS

BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

INPUT_FILE = DATA_DIR / "weblog.csv"
FILTERED_FILE = OUTPUT_DIR / "filtered_logs.csv"
SUMMARY_FILE = OUTPUT_DIR / "log_summary.json"
BACKUP_FILE = BACKUP_DIR / "log_summary_backup.json"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Directories created successfully.")
print("Dataset exists:", INPUT_FILE.exists())

Directories created successfully.
Dataset exists: True


In [32]:
# [CELL 4] INSPECT CSV DATASET AND DISPLAY SAMPLE RECORD

print("Dataset exists:", INPUT_FILE.exists())
print("Dataset path:", INPUT_FILE)

Dataset exists: True
Dataset path: /Users/jherielle/Downloads/CORREA/Lab6_File_Handling_Correa/Lab6_File_Handling_Additional_Exercises/data/weblog.csv


In [33]:
# [CELL 5] READ CSV LINE BY LINE AND FILTER HTTP STATUS CODES

status_counts = Counter()
url_counts = Counter()

records_processed = 0
records_in_target_class = 0
invalid_records = 0

with open(INPUT_FILE, "r", encoding="utf-8-sig", newline="") as file:

    reader = csv.DictReader(file)

    columns = {
        str(name).strip().lower(): name
        for name in reader.fieldnames
    }

    ip_col = columns.get("ip")
    time_col = columns.get("time")
    url_col = columns.get("url")
    status_col = columns.get("staus") or columns.get("status")

    if not all([ip_col, time_col, url_col, status_col]):
        raise ValueError("Required columns are missing.")

    with open(FILTERED_FILE, "w", encoding="utf-8", newline="") as output:

        fieldnames = ["IP", "Time", "URL", "Staus"]
        writer = csv.DictWriter(output, fieldnames=fieldnames)

        writer.writeheader()

        for row in reader:

            if records_processed >= RECORDS_TO_PROCESS:
                break

            records_processed += 1

            ip = (row.get(ip_col) or "").strip()
            timestamp = (row.get(time_col) or "").strip()
            url = (row.get(url_col) or "").strip()
            status = (row.get(status_col) or "").strip()

            if not all([ip, timestamp, url, status]):
                invalid_records += 1
                continue

            match = re.search(r"\b([1-5]\d{2})\b", status)

            if not match:
                invalid_records += 1
                continue

            status_code = int(match.group(1))
            status_class = status_code // 100

            if status_class == TARGET_STATUS_CLASS:

                records_in_target_class += 1
                status_counts[status_code] += 1
                url_counts[url] += 1

                writer.writerow({
                    "IP": ip,
                    "Time": timestamp,
                    "URL": url,
                    "Staus": status_code
                })

print("Log processing completed!")
print("Records processed:", records_processed)
print("Records in target class:", records_in_target_class)
print("Invalid records:", invalid_records)

Log processing completed!
Records processed: 1016
Records in target class: 5
Invalid records: 0


In [34]:
# [CELL 6] COUNT STATUS CODES AND IDENTIFY TOP FIVE URLS

top_five_urls = url_counts.most_common(5)

print("Count per status code:")

for code, count in sorted(status_counts.items()):
    print(code, ":", count)

print("\nFive most requested URLs:")

for url, count in top_five_urls:
    print(url, ":", count)

Count per status code:
404 : 5

Five most requested URLs:
GET /robots.txt HTTP/1.1 : 4
GET /favicon.ico HTTP/1.1 : 1


In [35]:
# [CELL 7] CREATE JSON PROCESSING SUMMARY

summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "records_processed": records_processed,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "records_in_target_class": records_in_target_class,
    "invalid_records": invalid_records,
    "count_per_status_code": dict(status_counts),
    "top_five_urls": [
        {"url": url, "count": count}
        for url, count in top_five_urls
    ]
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created successfully!")

JSON summary created successfully!


In [36]:
# [CELL 8] CREATE AND VERIFY JSON BACKUP

shutil.copy2(SUMMARY_FILE, BACKUP_FILE)

backup_verified = (
    BACKUP_FILE.exists()
    and SUMMARY_FILE.read_bytes() == BACKUP_FILE.read_bytes()
)

print("Backup exists:", BACKUP_FILE.exists())
print("Backup verified:", backup_verified)

Backup exists: True
Backup verified: True


In [37]:
# [CELL 9] DISPLAY FINAL REPORT AND VERIFY GENERATED FILES

with open(SUMMARY_FILE, "r", encoding="utf-8") as file:
    final_summary = json.load(file)

print("===== WEB LOG ANALYSIS REPORT =====")
print(json.dumps(final_summary, indent=4))

print("\n===== FILE VERIFICATION =====")
print("Filtered CSV exists:", FILTERED_FILE.exists())
print("JSON summary exists:", SUMMARY_FILE.exists())
print("Backup exists:", BACKUP_FILE.exists())
print("Backup verified:", backup_verified)

===== WEB LOG ANALYSIS REPORT =====
{
    "student": "Correa",
    "student_id": "TUPM-26-1106",
    "seed_digit": 6,
    "id_sum": 16,
    "records_processed": 1016,
    "target_status_class": "4xx",
    "records_in_target_class": 5,
    "invalid_records": 0,
    "count_per_status_code": {
        "404": 5
    },
    "top_five_urls": [
        {
            "url": "GET /robots.txt HTTP/1.1",
            "count": 4
        },
        {
            "url": "GET /favicon.ico HTTP/1.1",
            "count": 1
        }
    ]
}

===== FILE VERIFICATION =====
Filtered CSV exists: True
JSON summary exists: True
Backup exists: True
Backup verified: True
